In [12]:
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

load_dotenv()


True

In [ ]:
from langchain_openai import ChatOpenAI
import os

llm= ChatOpenAI(
    base_url=os.getenv("OCI_LLM_BASE_URL"),
    api_key=os.getenv("OCI_LLM_API_KEY"),
    model=os.getenv("MODEL_QWEN"),
    temperature=0.2
)

class State(TypedDict):
    messages: Annotated[list,add_messages]


def chatbot(state:State) -> State:
    return {"messages":[llm.invoke(state["messages"])]}

builder = StateGraph(State)
builder.add_node("chatbot_node", chatbot)

builder.add_edge(START, "chatbot_node")
builder.add_edge("chatbot_node", END)

graph = builder.compile()

In [14]:
message ={"role": "user", "content": "Who walked on the Antarctica for the fist time? Print only the name"}
response=graph.invoke({"messages":[message]})
response

{'messages': [HumanMessage(content='Who walked on the Antarctica for the fist time? Print only the name', additional_kwargs={}, response_metadata={}, id='56f4af50-1ec0-4c0b-be50-b8e1c3378e3a'),
  AIMessage(content='Robert Falcon Scott', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 4, 'prompt_tokens': 22, 'total_tokens': 26, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 3, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'Qwen3-30B-A3B-Instruct-2507-IQ4_NL.gguf', 'system_fingerprint': 'b12282-aa75ff323', 'id': 'chatcmpl-wAeTzTv6ReT5Ybrn7cBypsxkiOTOky70', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1e5fb-b3cd-72e9-a8c1-cc460042eef6-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 22, 'output_tokens': 4, 'total_tokens': 26, 'input_token_details': {'cache_read': 3}, 'output_token_detai

In [ ]:
state = None
while True:
    in_message = input("You: ")
    if in_message.lower() in {"quit","exit"}:
        break
    if state is None:
        state: State = {
            "messages": [{"role": "user", "content": in_message}]
        }
    else:
        state["messages"].append({"role": "user", "content": in_message})

    state = graph.invoke(state)
    print("Bot:", state["messages"][-1].content)